# Comb-content derived tables — generators

The generators for the derived statistics tables, in the package module
[`combinator/content/make.py`](combinator/content/make.py) (derived from the
paper's ``CombContent_Make.ipynb``).  These tables are the inputs to
`combinator.content.data`, which the analysis notebook
(`CombContent_Analysis.ipynb`) turns into the paper's Figures 1–5.

| Committed table | Generator | Runtime | Reproducible? |
|---|---|---|---|
| `df_cellneib.csv` | `cell_neighbour_table` | ~1 h | yes (deterministic) |
| `df_cellfuture.csv` | `cell_future_table` | ~25 min | yes (deterministic) |
| `df_nearestdist.csv`, `df_nearestdist_sum.csv` | `nearest_neighbour_table` | ~10 min | deterministic columns bit-for-bit; Monte-Carlo columns depend on the numpy build that drew them |
| `df_nearestdist_nest.csv`, `df_nearestdist_nest_sum.csv` | `nest_nearest_neighbour_table` | ~30 min | **no** — needs `df_mergedlandscape.csv`, never released (see `RESULTS.md`) |

Run the full build from the repository root with the CLI, e.g.:

    uv run python -m combinator.make_content_tables --tables nearestdist

The committed CSVs at the repository root are canonical; the subset runs
below demonstrate the generators without overwriting anything.


In [1]:
import numpy as np
import pandas as pd

from combinator.content import data, make

Contents = data.load_masks(keys=("Contents",))["Contents"]
ContentsCrop = data.load_masks(keys=("ContentsCrop",))["ContentsCrop"]
print("masks:", Contents.shape, ContentsCrop.shape,
      "| per colony:", Contents[0].shape)

masks: (12,) (12,) | per colony: (20, 20, 52, 100)


In [2]:
# Inventory of the committed tables (the outputs of this pipeline)
for name in ("df_cellneib", "df_cellfuture", "df_nearestdist",
             "df_nearestdist_sum", "df_nearestdist_nest",
             "df_nearestdist_nest_sum"):
    df = pd.read_csv(f"{name}.csv", nrows=1_000_000)
    print(f"{name:24s} {df.shape}")

df_cellneib              (239556, 16)
df_cellfuture            (163695, 15)
df_nearestdist           (16192, 18)
df_nearestdist_sum       (4143, 8)
df_nearestdist_nest      (1235, 16)
df_nearestdist_nest_sum  (236, 6)


## Subset re-runs (fast demo)

Re-derive the generators for colony DD1 only and check them against the
committed rows.  These must match **exactly** — the generators are pure
numpy/pandas given the same masks.


In [3]:
# df_cellneib: colony DD1, first 2 weeks (~20 s)
sub = np.empty(1, dtype=object)
sub[0] = Contents[0][:2]
df = make.cell_neighbour_table(sub)

ref = pd.read_csv("df_cellneib.csv")
ref = ref[(ref.colony == "DD1") & (ref.week <= 2)].reset_index(drop=True)
df = df.reset_index(drop=True)
same = (df.shape == ref.shape and
        list(df.columns) == list(ref.columns) and
        np.allclose(df.select_dtypes("number").values,
                    ref.select_dtypes("number").values,
                    rtol=1e-12, equal_nan=True) and
        (df.select_dtypes("object").values ==
         ref.select_dtypes("object").values).all())
print("cell_neighbour_table subset:", df.shape, "| matches committed:", same)

DD1
cell_neighbour_table subset: (1620, 16) | matches committed: True


<cell 4>:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
<cell 4>:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.


In [4]:
# df_cellfuture: colony DD1, first 2 weeks
sub[0] = Contents[0][:3]
df = make.cell_future_table(sub)
df = df[df.week <= 2].reset_index(drop=True)

ref = pd.read_csv("df_cellfuture.csv")
ref = ref[(ref.colony == "DD1") & (ref.week <= 2)].reset_index(drop=True)
same = (df.shape == ref.shape and list(df.columns) == list(ref.columns))
print("cell_future_table subset:", df.shape, "| matches committed:", same)

DD1
cell_future_table subset: (1141, 15) | matches committed: True


In [5]:
# df_nearestdist: colony DD1, week 1 only (~5 s of the ~10 min full run)
sub = np.empty(1, dtype=object)
sub[0] = ContentsCrop[0][:1]
df, df_sum = make.nearest_neighbour_table(sub, seed=42)

ref = pd.read_csv("df_nearestdist.csv")
ref = ref[(ref.colony == "DD1") & (ref.week == 1)]
df = df.reset_index(drop=True)
ok_counts = np.array_equal(df["count"].values, ref["count"].values)
ok_obs = np.allclose(df["nearestneibavg"], ref["nearestneibavg"])
print("nearest_neighbour_table subset:", df.shape,
      "| deterministic columns match committed:", ok_counts and ok_obs)

colony: DD1, week: 01
nearest_neighbour_table subset: (50, 18) | deterministic columns match committed: True


The full nest-level tables (`df_nearestdist_nest*.csv`) cannot be
reproduced from the released data: their generator
(`make.nest_nearest_neighbour_table`, provided for completeness) requires
`df_mergedlandscape.csv`, the per-frame landscape table that was not
released with the paper. See `RESULTS.md`.
